# Transcript → time ranges → requirements (starter)

Grows one column at a time. The table stays **one row per turn**; we keep adding
columns to the right:

1. `time_range`, `speaker`, `text`  — parse the transcript (no pipeline needed)
2. `context`   — one or more time ranges **combined into a chunk** sized to fit the LLM's context window
3. `requirements` / `inferred_types` / `confidences` — what the derivation returned for that turn

Steps 1 & 3-slice are pure `pandas`. From step 4 on we drive the **real** SpecLive
pipeline (`AnalysisService` + provider + context strategy), so `context` is
literally the chunk the model saw and the requirements are literally what it
produced.

## 1 · Config — pick a transcript

In [1]:
from pathlib import Path
import json
import pandas as pd

_here = Path.cwd()
REPO = next(p for p in (_here, *_here.parents) if (p / "research" / "corpus").exists())

CASE = "hospital-bed-mgmt"    # any dir under research/corpus
TRANSCRIPT = REPO / "research" / "corpus" / CASE / "transcript.json"

WORDS_PER_MINUTE = 150        # speaking pace → estimated durations
GAP_SECONDS = 0.5             # pause between turns

# pipeline knobs (used from step 2 on)
PROVIDER = "mock"             # "mock" (no keys) | "openai_compatible" (needs LLM_API_KEY)
STRATEGY = "window"          # "segment" | "window" | "full"  — how many time ranges get combined
WINDOW_SECONDS = 90          # STRATEGY == "window": ~this many seconds of convo per chunk
MODEL_CONTEXT_TOKENS = 128_000   # the context window of the LLM we're targeting

pd.set_option("display.max_colwidth", 70)
print("transcript:", TRANSCRIPT.relative_to(REPO))

transcript: research/corpus/hospital-bed-mgmt/transcript.json


## 2 · Parse into a time-ranged table

One row per turn: `start` / `end` (seconds), a human `time_range`, speaker, text. Timestamps are estimated from word count unless the turn carries real ones.

In [2]:
def mmss(seconds: float) -> str:
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"


def load_turns(path: Path) -> pd.DataFrame:
    data = json.loads(Path(path).read_text())
    turns = data["turns"] if isinstance(data, dict) else data
    rows, clock = [], 0.0
    for i, turn in enumerate(turns):
        text = (turn.get("text") or "").strip()
        n = len(text.split())
        if turn.get("start_time") is not None and turn.get("end_time") is not None:
            start, end = float(turn["start_time"]), float(turn["end_time"])
        else:
            start = clock
            end = start + n / WORDS_PER_MINUTE * 60.0
            clock = end + GAP_SECONDS
        rows.append({
            "turn": i, "start": round(start, 1), "end": round(end, 1),
            "time_range": f"{mmss(start)}–{mmss(end)}",
            "speaker": turn.get("speaker_name") or turn.get("speaker") or "unknown",
            "words": n, "text": text,
        })
    return pd.DataFrame(rows)


df = load_turns(TRANSCRIPT)
print(f"{len(df)} turns · ~{mmss(df['end'].max())} total")
df[["time_range", "speaker", "text"]]

8 turns · ~5:37 total


,time_range,speaker,text
0,0:00–0:39,Dr. Naomi Fields,"Good morning, everyone. Today I want to walk you through a real pr..."
1,0:40–1:23,Dr. Naomi Fields,"Let me describe how bed assignment actually works today, because i..."
2,1:24–1:58,Dr. Naomi Fields,The exceptions are where it really falls apart. Isolation patients...
3,1:58–2:36,Dr. Naomi Fields,"So here's what I need this system to actually move. First, average..."
4,2:37–3:12,Dr. Naomi Fields,"On the technical side, everything has to integrate with our existi..."
5,3:13–4:00,Dr. Naomi Fields,"Now, constraints, and there are some real ones. This is a HIPAA en..."
6,4:01–4:42,Dr. Naomi Fields,"A couple of things I genuinely don't have answers to yet, and I wa..."
7,4:43–5:37,Dr. Naomi Fields,"Let me just play back what I've described, because I want you to p..."


## 3 · Slice by time

In [3]:
def between(df, start_s, end_s):
    """Turns overlapping the [start_s, end_s) window."""
    return df[(df["end"] > start_s) & (df["start"] < end_s)]

between(df, 0, 60)[["time_range", "speaker", "text"]]

,time_range,speaker,text
0,0:00–0:39,Dr. Naomi Fields,"Good morning, everyone. Today I want to walk you through a real pr..."
1,0:40–1:23,Dr. Naomi Fields,"Let me describe how bed assignment actually works today, because i..."


## 4 · Run the real derivation

Ingest the transcript into a live session and run `AnalysisService` — the same
code the app runs. Segments come back in turn order, so `segments[i]` is turn `i`
and we can join results back onto `df` by turn index.

In [4]:
import sys
HARNESS = str((REPO / "research" / "harness").resolve())
if HARNESS not in sys.path:
    sys.path.insert(0, HARNESS)

import ingest as ingest_mod
from _bootstrap import make_context, provider_available
from app.services.context_strategy import get_context_strategy

ok, why = provider_available(PROVIDER)
assert ok, f"provider {PROVIDER!r} unavailable: {why}"

transcript = ingest_mod.load_transcript(TRANSCRIPT)
ctx = make_context(PROVIDER, STRATEGY)
try:
    # same WPM clock as load_turns, so context_range lines up with time_range
    segments = ingest_mod.ingest(ctx, transcript,
                                 words_per_minute=WORDS_PER_MINUTE, gap_seconds=GAP_SECONDS)
    ctx.analysis.analyze_session(ctx.session_id)
    derived = ctx.artifacts.list_for_session(ctx.session_id)
    evidence_by_artifact = {a.id: ctx.artifacts.evidence_for(a.id) for a in derived}
finally:
    ctx.close()

print(f"{len(derived)} artifacts derived from {len(segments)} turns")

6 artifacts derived from 8 turns


## 5 · Chunk the transcript into `context` windows

**`context` = one or more time ranges combined into a single chunk sent to the
LLM.** The `STRATEGY` knob decides how much gets combined — the whole point is
to stay inside the model's context window:

- `segment` → one turn per chunk (smallest prompt, no cross-turn context)
- `window`  → ~`WINDOW_SECONDS` of conversation per chunk (e.g. a minute)
- `full`    → the whole conversation in one chunk

`build_context_chunks` shows each chunk's combined `context_range`, how many
turns it merges, and its approximate token count — which we check against
`MODEL_CONTEXT_TOKENS`.

In [5]:
import importlib
import frames
importlib.reload(frames)

strategy = get_context_strategy(STRATEGY, window_seconds=WINDOW_SECONDS)

chunks = frames.build_context_chunks(segments, strategy)
chunks["within_budget"] = chunks["context_tokens"] <= MODEL_CONTEXT_TOKENS
print(f"STRATEGY={STRATEGY!r} → {len(chunks)} chunk(s); "
      f"largest ≈{chunks['context_tokens'].max()} tokens vs "
      f"{MODEL_CONTEXT_TOKENS:,} budget "
      f"({'all fit' if chunks['within_budget'].all() else 'OVER BUDGET — shrink the window'})")
chunks[["chunk_id", "context_range", "n_turns", "context_tokens", "within_budget", "text"]]

STRATEGY='window' → 3 chunk(s); largest ≈458 tokens vs 128,000 budget (all fit)


,chunk_id,context_range,n_turns,context_tokens,within_budget,text
0,0,0:00–1:58,3,445,True,"Dr. Naomi Fields: Good morning, everyone. Today I want to walk you..."
1,1,1:58–4:00,3,458,True,Dr. Naomi Fields: So here's what I need this system to actually mo...
2,2,4:01–5:37,2,360,True,Dr. Naomi Fields: A couple of things I genuinely don't have answer...


## 6 · Add `requirements`, `inferred_types`, `confidences`

Now join what the derivation returned onto the time-range table, per turn, plus
the chunk each turn was sent in (`context_range`, `context_tokens`).

In [6]:
seg_view = frames.build_segment_coverage(
    derived=derived, evidence_by_artifact=evidence_by_artifact,
    segments=segments, context_strategy=strategy,
)

cols = ["chunk_id", "context_range", "context_tokens",
        "requirements", "inferred_types", "confidences", "inferred"]
table = df.merge(
    seg_view[["segment_seq", *cols]].rename(columns={"segment_seq": "turn"}),
    on="turn", how="left",
)

print(f"{table['inferred'].sum()}/{len(table)} turns produced a requirement")
table[["time_range", "text", "chunk_id", "context_range", "context_tokens",
       "requirements", "inferred_types", "confidences"]]

4/8 turns produced a requirement


,time_range,text,chunk_id,context_range,context_tokens,requirements,inferred_types,confidences
0,0:00–0:39,"Good morning, everyone. Today I want to walk you through a real pr...",0,0:00–1:58,445,[],[],[]
1,0:40–1:23,"Let me describe how bed assignment actually works today, because i...",0,0:00–1:58,445,[],[],[]
2,1:24–1:58,The exceptions are where it really falls apart. Isolation patients...,0,0:00–1:58,445,[],[],[]
3,1:58–2:36,"So here's what I need this system to actually move. First, average...",1,1:58–4:00,458,[The solution shall be delivered within the stated deployment dead...,[constraint],[0.77]
4,2:37–3:12,"On the technical side, everything has to integrate with our existi...",1,1:58–4:00,458,[The solution shall operate on the existing field devices already ...,"[constraint, integration]","[0.82, 0.66]"
5,3:13–4:00,"Now, constraints, and there are some real ones. This is a HIPAA en...",1,1:58–4:00,458,[The solution shall integrate with the identified source system(s).],[integration],[0.62]
6,4:01–4:42,"A couple of things I genuinely don't have answers to yet, and I wa...",2,4:01–5:37,360,[],[],[]
7,4:43–5:37,"Let me just play back what I've described, because I want you to p...",2,4:01–5:37,360,[The solution shall integrate with the identified source system(s)...,"[integration, success_metric]","[0.66, 0.63]"


In [7]:
# just the turns that produced something, exploded one requirement per row
hits = table[table["inferred"]].explode(
    ["requirements", "inferred_types", "confidences"]
)
hits[["time_range", "context_range", "inferred_types", "confidences", "requirements"]]

,time_range,context_range,inferred_types,confidences,requirements
3,1:58–2:36,1:58–4:00,constraint,0.77,The solution shall be delivered within the stated deployment deadl...
4,2:37–3:12,1:58–4:00,constraint,0.82,The solution shall operate on the existing field devices already i...
4,2:37–3:12,1:58–4:00,integration,0.66,The solution shall integrate with the identified source system(s).
5,3:13–4:00,1:58–4:00,integration,0.62,The solution shall integrate with the identified source system(s).
7,4:43–5:37,4:01–5:37,integration,0.66,The solution shall integrate with the identified source system(s).
7,4:43–5:37,4:01–5:37,success_metric,0.63,Success shall be measured by the stated operational metric.


## From here

The table now reads left→right as the whole inference story for each turn:
**when → what was said → what the LLM saw → what it inferred → how sure.**

Natural next columns (all live in `frames.build_ledger` / `detect_dependencies`
already, see `inference_explorer.ipynb`):

- `evidence_quote` — the exact words the requirement was grounded in
- `dependency_kind` — reinforcement vs. conflict with other turns
- gold columns (`correct_type`, …) for scoring against ground truth

Swap `PROVIDER = "openai_compatible"` (with `LLM_API_KEY` set) to see real-model
requirements instead of the keyword mock, and try `STRATEGY = "full"` vs
`"segment"` to watch the `context` column grow or shrink.